# Attention by head for one seeded random input

This notebook samples one input deterministically from `DATA_SEED`, then shows every attention head in every layer as an interactive key-by-query heatmap. Columns are query positions and rows are key positions. Hover over a cell to see its query token, key token, and exact attention weight.

Change `DATA_SEED` and run all cells to inspect another random input. The optional query/key bounds can zoom the plots without changing the sampled sequence.

In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from IPython.display import display

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"

# Change this value and rerun the notebook for another random input.
DATA_SEED = 15_002

# Optional plot window. None means the end of the sequence.
QUERY_START = 0
QUERY_END = None
KEY_START = 0
KEY_END = None
TICK_EVERY = 10

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(
    path=CHECKPOINT_PATH, model=model, map_location=device
)
model.eval()

n_layers = cfg["model"]["n_layers"]
n_heads = cfg["model"]["n_heads"]
seq_len = cfg["task"]["seq_len"]
query_end = seq_len if QUERY_END is None else QUERY_END
key_end = seq_len if KEY_END is None else KEY_END
if not 0 <= QUERY_START < query_end <= seq_len:
    raise ValueError("Query bounds must define a nonempty range inside the sequence.")
if not 0 <= KEY_START < key_end <= seq_len:
    raise ValueError("Key bounds must define a nonempty range inside the sequence.")
if TICK_EVERY <= 0:
    raise ValueError("TICK_EVERY must be positive.")

print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Model: {n_layers} layers x {n_heads} heads; sequence length {seq_len}")
print(f"Data seed: {DATA_SEED}")

Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Model: 3 layers x 4 heads; sequence length 150
Data seed: 15002


## Seeded input

The table provides the position-to-token lookup used by the heatmaps. `character_count_after_input` is the cumulative count through that input token and resets to zero at each newline.

In [3]:
batch = project_data.make_batch(
    batch_size=1, vocab=vocab, task_cfg=cfg["task"],
    device=device, seed=DATA_SEED,
)
input_ids = batch.tokens[0].detach().cpu().tolist()
target_ids = batch.targets[0].detach().cpu().tolist()
input_tokens = [vocab.decode_token(token_id) for token_id in input_ids]
target_tokens = [vocab.decode_token(token_id) for token_id in target_ids]

running_count = 0
character_counts = []
for token_id, token_name in zip(input_ids, input_tokens):
    if token_id == vocab.newline_id:
        running_count = 0
    else:
        running_count += project_data.extract_character_count(token_name)
    character_counts.append(running_count)

token_table = pd.DataFrame({
    "position": np.arange(seq_len),
    "input_token": input_tokens,
    "target_next_token": target_tokens,
    "character_count_after_input": character_counts,
})
print(f"Sampled line width: {batch.line_lengths[0].item()}")
with pd.option_context("display.max_rows", seq_len):
    display(token_table.set_index("position"))

Sampled line width: 52


,input_token,target_next_token,character_count_after_input
position,,,
0,BOS,TOKEN_7_4,0
1,TOKEN_7_4,TOKEN_1_7,4
2,TOKEN_1_7,TOKEN_8_5,11
3,TOKEN_8_5,TOKEN_9_8,16
4,TOKEN_9_8,TOKEN_2_2,24
5,TOKEN_2_2,TOKEN_4_6,26
6,TOKEN_4_6,TOKEN_5_2,32
7,TOKEN_5_2,TOKEN_3_4,34
8,TOKEN_3_4,TOKEN_7_3,38


## Collect every attention pattern

TransformerLens attention patterns have shape `[batch, head, query, key]`. The diagnostic below confirms that each full query row sums to one up to floating-point error.

In [4]:
pattern_names = [
    f"blocks.{layer}.attn.hook_pattern" for layer in range(n_layers)
]
with torch.no_grad():
    _, cache = model.run_with_cache(
        batch.tokens, prepend_bos=False,
        names_filter=lambda name: name in pattern_names,
    )

attention_patterns = torch.stack([
    cache[name][0].detach().cpu() for name in pattern_names
])
assert attention_patterns.shape == (n_layers, n_heads, seq_len, seq_len)
row_sum_error = (attention_patterns.sum(dim=-1) - 1.0).abs().max().item()
print("Attention tensor [layer, head, query, key]:", tuple(attention_patterns.shape))
print(f"Maximum full-row normalization error: {row_sum_error:.3e}")

Attention tensor [layer, head, query, key]: (3, 4, 150, 150)
Maximum full-row normalization error: 2.384e-07


## Interactive key-by-query attention maps

Each layer has one panel per head and a shared color scale. Query position increases to the right; key position increases downward. Causally forbidden future-key cells are zero. Hovering a cell reports both position/token labels and the exact attention value.

In [5]:
def position_token_labels(start: int, end: int) -> list[str]:
    return [f"{position}: {input_tokens[position]}" for position in range(start, end)]


def tick_labels(labels: list[str], every: int) -> list[str]:
    selected = labels[::every]
    if labels[-1] not in selected:
        selected = [*selected, labels[-1]]
    return selected


query_labels = position_token_labels(QUERY_START, query_end)
key_labels = position_token_labels(KEY_START, key_end)
query_ticks = tick_labels(query_labels, TICK_EVERY)
key_ticks = tick_labels(key_labels, TICK_EVERY)
n_columns = min(2, n_heads)
n_rows = math.ceil(n_heads / n_columns)

for layer in range(n_layers):
    layer_attention = attention_patterns[
        layer, :, QUERY_START:query_end, KEY_START:key_end
    ].numpy()
    layer_max = max(float(layer_attention.max()), 1e-12)
    fig = make_subplots(
        rows=n_rows, cols=n_columns,
        subplot_titles=[f"Layer {layer}, head {head}" for head in range(n_heads)],
        horizontal_spacing=0.08, vertical_spacing=0.10,
    )

    for head in range(n_heads):
        row = head // n_columns + 1
        column = head % n_columns + 1
        fig.add_trace(
            go.Heatmap(
                z=layer_attention[head].T, x=query_labels, y=key_labels,
                coloraxis="coloraxis", zsmooth=False,
                hovertemplate=(
                    "query=%{x}<br>key=%{y}<br>attention=%{z:.6f}<extra></extra>"
                ),
            ),
            row=row, col=column,
        )

    fig.update_xaxes(
        title_text="Query position: token", tickmode="array",
        tickvals=query_ticks, ticktext=query_ticks, tickangle=-45,
    )
    fig.update_yaxes(
        title_text="Key position: token", tickmode="array",
        tickvals=key_ticks, ticktext=key_ticks, autorange="reversed",
    )
    fig.update_layout(
        title=(
            f"Layer {layer} attention for seed {DATA_SEED}<br>"
            f"<sup>queries {QUERY_START}:{query_end}; keys {KEY_START}:{key_end}</sup>"
        ),
        width=1300, height=max(820, 440 * n_rows),
        coloraxis={
            "colorscale": "Viridis", "cmin": 0.0, "cmax": layer_max,
            "colorbar": {"title": "Attention"},
        },
        hovermode="closest",
    )
    fig.show()